In [1]:
import json
from collections import Counter
from pathlib import Path
from typing import Any


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


chunks = load_jsonl(
    Path("data/processed/chunks/sec_10k_chunks.jsonl")
)
proxies = load_jsonl(
    Path(
        "data/processed/table_proxies/"
        "sec_10k_table_proxies.jsonl"
    )
)
records = load_jsonl(
    Path(
        "data/processed/embedding_records/"
        "sec_10k_embedding_records.jsonl"
    )
)

chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}
proxies_by_target = {
    proxy["target_chunk_id"]: proxy
    for proxy in proxies
}

type_counts = Counter(
    record["record_type"]
    for record in records
)

assert type_counts == {"text": 3890, "table": 3220}
assert len(records) == 7110
assert len({record["record_id"] for record in records}) == 7110

for record in records:
    assert record["record_type"] in {"text", "table"}
    assert record["embedding_text"].strip()
    assert record["document"].strip()
    assert record["target_chunk_id"] in chunks_by_id

    assert all(
        isinstance(value, (str, int, float, bool))
        for value in record["metadata"].values()
    )

    chunk = chunks_by_id[record["target_chunk_id"]]

    assert record["document"] == chunk["text"]
    assert "SEC item: None" not in record["embedding_text"]
    assert "Section: None" not in record["embedding_text"]

    if record["record_type"] == "text":
        assert record["record_id"] == f"text::{chunk['chunk_id']}"
        assert record["embedding_text"].endswith(chunk["text"])

    else:
        proxy = proxies_by_target[record["target_chunk_id"]]

        assert record["record_id"] == f"table::{proxy['proxy_id']}"
        assert record["embedding_text"] == proxy["proxy_text"]
        assert chunk["element_type"] == "table"

print("All embedding record checks passed!")
print(f"Record types: {dict(type_counts)}")
print(
    "Banks:",
    sorted({record["metadata"]["ticker"] for record in records}),
)

All embedding record checks passed!
Record types: {'text': 3890, 'table': 3220}
Banks: ['ALLY', 'BAC', 'C', 'GS', 'JPM', 'LOB', 'PNC', 'TFC', 'USB', 'WFC']


In [2]:
sample_specs = [
    ("text", "JPM"),
    ("text", "BAC"),
    ("table", "GS"),
    ("table", "PNC"),
]

for record_type, ticker in sample_specs:
    record = next(
        record
        for record in records
        if record["record_type"] == record_type
        and record["metadata"]["ticker"] == ticker
    )

    print("\n" + "=" * 80)
    print(f"{record_type.upper()} | {ticker}")
    print(f"Record ID: {record['record_id']}")
    print(f"Target: {record['target_chunk_id']}")
    print("\nEMBEDDING TEXT:")
    print(record["embedding_text"][:600])
    print("\nORIGINAL DOCUMENT:")
    print(record["document"][:600])


TEXT | JPM
Record ID: text::3ec9ac64c56f27219a941816e88fb3f0b1ae37663d4a8f7393731a569887444a
Target: 3ec9ac64c56f27219a941816e88fb3f0b1ae37663d4a8f7393731a569887444a

EMBEDDING TEXT:
Bank: JPM
Report: 2025 10-K
Section: UNITED STATES

UNITED STATES

ORIGINAL DOCUMENT:
UNITED STATES

TEXT | BAC
Record ID: text::ebbe0668b4c80776808514adcac7cbc19acb45034e6f16b8b1b146e1ff0a6e12
Target: ebbe0668b4c80776808514adcac7cbc19acb45034e6f16b8b1b146e1ff0a6e12

EMBEDDING TEXT:
Bank: BAC
Report: 2025 10-K
Section: UNITED STATES

UNITED STATES

ORIGINAL DOCUMENT:
UNITED STATES

TABLE | GS
Record ID: table::abd0057ebc2b6ad047fba8b90c28ffce0572a2ab6203cb70144165885713f1a3
Target: cad8ad42202cf7b56b07637566a43caa887ffe5fe03b7963663de2380b6261c2

EMBEDDING TEXT:
Bank: GS
Report: 2025 10-K
Section: THE SECURITIES EXCHANGE ACT OF 1934
Columns: For the fiscal year ended December 31 , 2025; Commission File Number: 001-14965

ORIGINAL DOCUMENT:
For the fiscal year ended December 31 , 2025 |  | Commission File 

In [3]:
from collections import Counter


def document_word_count(record: dict) -> int:
    return len(record["document"].split())


thresholds = (5, 10, 25, 50)

for record_type in ("text", "table"):
    type_records = [
        record
        for record in records
        if record["record_type"] == record_type
    ]

    print(f"\n{record_type.upper()}: {len(type_records)}")

    for threshold in thresholds:
        count = sum(
            document_word_count(record) <= threshold
            for record in type_records
        )
        percentage = 100 * count / len(type_records)

        print(
            f"At most {threshold:>2} words: "
            f"{count} ({percentage:.1f}%)"
        )

    shortest_sections = Counter(
        str(record["metadata"].get("section_title", ""))
        for record in type_records
        if document_word_count(record) <= 10
    )

    print("Most common sections among records with at most 10 words:")

    for section, count in shortest_sections.most_common(10):
        print(f"  {count:>4} | {section or '<missing>'}")


TEXT: 3890
At most  5 words: 213 (5.5%)
At most 10 words: 367 (9.4%)
At most 25 words: 682 (17.5%)
At most 50 words: 1027 (26.4%)
Most common sections among records with at most 10 words:
    33 | Item 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA
    22 | FORWARD-LOOKING STATEMENTS
    17 | THE GOLDMAN SACHS GROUP, INC. AND SUBSIDIARIES
    16 | SECURITIES AND EXCHANGE COMMISSION
    14 | Item 8. Financial Statements and Supplementary Data
    11 | Item 6. [Reserved]
     9 | UNITED STATES
     9 | Item 8. Financial Statements and Supplementary Data Table of Contents
     8 | FORM 10-K
     6 | Item 4. Mine Safety Disclosures

TABLE: 3220
At most  5 words: 44 (1.4%)
At most 10 words: 764 (23.7%)
At most 25 words: 807 (25.1%)
At most 50 words: 935 (29.0%)
Most common sections among records with at most 10 words:
   239 | THE GOLDMAN SACHS GROUP, INC. AND SUBSIDIARIES
   169 | FORWARD-LOOKING STATEMENTS
    63 | Item 6. [Reserved]
    54 | Item 8. Financial Statements and Supplementar

In [4]:
sample_specs = [
    ("text", "JPM", 80),
    ("text", "BAC", 80),
    ("table", "GS", 30),
    ("table", "PNC", 30),
]

for record_type, ticker, minimum_words in sample_specs:
    candidates = [
        record
        for record in records
        if record["record_type"] == record_type
        and record["metadata"]["ticker"] == ticker
        and document_word_count(record) >= minimum_words
    ]

    record = candidates[len(candidates) // 2]

    print("\n" + "=" * 80)
    print(f"{record_type.upper()} | {ticker}")
    print("Section:", record["metadata"].get("section_title"))
    print("Document words:", document_word_count(record))

    print("\nEMBEDDING TEXT:")
    print(record["embedding_text"][:800])

    print("\nORIGINAL DOCUMENT:")
    print(record["document"][:800])


TEXT | JPM
Section: CRITICAL ACCOUNTING ESTIMATES USED BY THE FIRM
Document words: 337

EMBEDDING TEXT:
Bank: JPM
Report: 2025 10-K
SEC item: Item 15
Section: CRITICAL ACCOUNTING ESTIMATES USED BY THE FIRM

For the year ended December 31, 2025, the Firm reviewed current economic conditions, estimated market cost of equity, as well as actual business results and projections of business performance. Based on such reviews, the Firm has concluded that goodwill was not impaired as of December 31, 2025. For each of the reporting units, fair value exceeded carrying value by at least 20% and there was no indication of a significant risk of goodwill impairment based on current projections and valuations.

The projections for the Firm’s reporting units are consistent with management’s current business outlook assumptions in the short term, and the Firm’s best estimates of long-term growth and return o

ORIGINAL DOCUMENT:
For the year ended December 31, 2025, the Firm reviewed current economic c

In [5]:
from collections import Counter
from time import perf_counter

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 4


def embedding_word_count(record: dict) -> int:
    return len(record["embedding_text"].split())


# Select one representative record per bank and record type.
smoke_records_by_id = {}

tickers = sorted(
    {
        record["metadata"]["ticker"]
        for record in records
    }
)

for ticker in tickers:
    for record_type in ("text", "table"):
        candidates = sorted(
            (
                record
                for record in records
                if record["record_type"] == record_type
                and record["metadata"]["ticker"] == ticker
            ),
            key=embedding_word_count,
        )

        if candidates:
            representative = candidates[len(candidates) // 2]
            smoke_records_by_id[representative["record_id"]] = representative

# Add the longest records to test memory usage and long inputs.
for record in sorted(
    records,
    key=embedding_word_count,
    reverse=True,
)[:8]:
    smoke_records_by_id[record["record_id"]] = record

smoke_records = list(smoke_records_by_id.values())
embedding_texts = [
    record["embedding_text"]
    for record in smoke_records
]

print(f"Smoke records: {len(smoke_records)}")
print(
    "Record types:",
    dict(Counter(record["record_type"] for record in smoke_records)),
)
print(
    "Banks:",
    sorted(
        {
            record["metadata"]["ticker"]
            for record in smoke_records
        }
    ),
)
print(
    "Embedding words:",
    f"min={min(map(embedding_word_count, smoke_records))},",
    f"max={max(map(embedding_word_count, smoke_records))}",
)

model = SentenceTransformer(MODEL_NAME)

if model.device.type == "cuda":
    model.half()
    torch.cuda.reset_peak_memory_stats()

print(f"Device: {model.device}")
print(f"Maximum sequence length: {model.max_seq_length}")

started_at = perf_counter()

embeddings = model.encode_document(
    embedding_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

elapsed_seconds = perf_counter() - started_at
embeddings = np.asarray(embeddings, dtype=np.float32)
embedding_dimension = model.get_sentence_embedding_dimension()

assert embedding_dimension is not None
assert embeddings.shape == (
    len(smoke_records),
    embedding_dimension,
)
assert embedding_dimension == 1024
assert np.isfinite(embeddings).all()

norms = np.linalg.norm(embeddings, axis=1)

assert np.allclose(norms, 1.0, atol=1e-5)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(
    "Norms:",
    f"min={norms.min():.6f},",
    f"mean={norms.mean():.6f},",
    f"max={norms.max():.6f}",
)
print(f"Time: {elapsed_seconds:.1f} s")

if model.device.type == "cuda":
    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
    print(f"Peak allocated GPU memory: {peak_memory_gb:.2f} GB")

print("Qwen document embedding smoke test passed!")

c:\venvs\bankscope\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Smoke records: 28
Record types: {'text': 18, 'table': 10}
Banks: ['ALLY', 'BAC', 'C', 'GS', 'JPM', 'LOB', 'PNC', 'TFC', 'USB', 'WFC']
Embedding words: min=47, max=627


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2123.89it/s]


Device: cpu
Maximum sequence length: 32768


Batches: 100%|██████████| 7/7 [14:13<00:00, 121.93s/it]
C:\Users\nikola.bakic\AppData\Local\Temp\ipykernel_20288\1866589309.py:98: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


AssertionError: 

In [6]:
norm_deviations = np.abs(norms - 1.0)
worst_indices = np.argsort(norm_deviations)[-5:][::-1]

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(
    "Norms:",
    f"min={norms.min():.8f},",
    f"mean={norms.mean():.8f},",
    f"max={norms.max():.8f}",
)
print(f"Maximum deviation: {norm_deviations.max():.8f}")

print("\nLargest deviations:")

for index in worst_indices:
    record = smoke_records[index]

    print(
        f"{index:>2} | "
        f"norm={norms[index]:.8f} | "
        f"type={record['record_type']} | "
        f"bank={record['metadata']['ticker']} | "
        f"words={embedding_word_count(record)}"
    )

assert embeddings.shape == (len(smoke_records), 1024)
assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
assert np.all(norms > 0)

print("\nBasic embedding checks passed!")

Shape: (28, 1024)
Dtype: float32
Norms: min=0.99844474, mean=1.00030828, max=1.00207877
Maximum deviation: 0.00207877

Largest deviations:
14 | norm=1.00207877 | type=text | bank=TFC | words=320
 0 | norm=1.00193274 | type=text | bank=ALLY | words=162
11 | norm=1.00184059 | type=table | bank=LOB | words=71
20 | norm=1.00177896 | type=text | bank=GS | words=627
 5 | norm=1.00165546 | type=table | bank=C | words=76

Basic embedding checks passed!


In [7]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer


project_root = Path.cwd()

embedding_path = (
    project_root
    / "data/processed/embeddings/qwen3_embedding_0_6b_records.npz"
)
records_path = (
    project_root
    / "data/processed/embedding_records/sec_10k_embedding_records.jsonl"
)


def as_text(value: object) -> str:
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


records = [
    json.loads(line)
    for line in records_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

with np.load(embedding_path, allow_pickle=False) as data:
    embeddings = data["embeddings"]
    record_ids = [as_text(value) for value in data["record_ids"]]
    model_name = as_text(data["model_name"].item())

assert record_ids == [record["record_id"] for record in records]

model = SentenceTransformer(model_name)


def retrieve(
    query: str,
    ticker: str | None = None,
    k: int = 5,
) -> list[dict]:
    query_embedding = model.encode_query(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].astype(np.float32)

    scores = embeddings @ query_embedding

    candidate_indices = np.array(
        [
            index
            for index, record in enumerate(records)
            if ticker is None
            or record["metadata"]["ticker"].upper() == ticker.upper()
        ]
    )

    ranked_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1][:k]
    ]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        record = records[index]
        metadata = record["metadata"]

        result = {
            "rank": rank,
            "score": float(scores[index]),
            "record_id": record["record_id"],
            "target_chunk_id": record["target_chunk_id"],
            "record_type": record["record_type"],
            "ticker": metadata["ticker"],
            "sec_item": metadata.get("sec_item"),
            "section_title": metadata.get("section_title"),
        }
        results.append(result)

        print(f"\n{'=' * 80}")
        print(result)
        print("\nEmbedding text:")
        print(record["embedding_text"][:600])
        print("\nOriginal document:")
        print(record["document"][:1200])

    return results

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3563.80it/s]


In [8]:
retrieve(
    query=(
        "What operational risks does JPMorgan face from cyberattacks "
        "and failures of technology systems?"
    ),
    ticker="JPM",
)


{'rank': 1, 'score': 0.8119903802871704, 'record_id': 'text::bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27', 'target_chunk_id': 'bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27', 'record_type': 'text', 'ticker': 'JPM', 'sec_item': 'Item 1A', 'section_title': 'Item 1A. Risk Factors.'}

Embedding text:
Bank: JPM
Report: 2025 10-K
SEC item: Item 1A
Section: Item 1A. Risk Factors.

JPMorganChase’s interconnectivity with clients, customers and other external parties continues to expand, which increases the risk of failure or cyber attacks with respect to the systems of those parties. Any systems failure, security breach, or human error or misconduct that affects clients, customers or external parties could require JPMorganChase to take steps to protect the integrity of its own operational systems or to safeguard confidential information, including restricting the access of its customers to thei

Original document:
JPMorganChase’s interconnectivity with c

[{'rank': 1,
  'score': 0.8119903802871704,
  'record_id': 'text::bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27',
  'target_chunk_id': 'bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27',
  'record_type': 'text',
  'ticker': 'JPM',
  'sec_item': 'Item 1A',
  'section_title': 'Item 1A. Risk Factors.'},
 {'rank': 2,
  'score': 0.7756694555282593,
  'record_id': 'text::07dd574d2143e30c76baeecdcc95bb07cedd0445d1141e04af61febb415c0d17',
  'target_chunk_id': '07dd574d2143e30c76baeecdcc95bb07cedd0445d1141e04af61febb415c0d17',
  'record_type': 'text',
  'ticker': 'JPM',
  'sec_item': 'Item 1A',
  'section_title': 'Item 1A. Risk Factors.'},
 {'rank': 3,
  'score': 0.76755291223526,
  'record_id': 'text::8f106940a9c6145cab20451d0dfc5a2b259b3dc95035700121a856261cb1062c',
  'target_chunk_id': '8f106940a9c6145cab20451d0dfc5a2b259b3dc95035700121a856261cb1062c',
  'record_type': 'text',
  'ticker': 'JPM',
  'sec_item': 'Item 1A',
  'section_title': 'Item 1A. Risk Fac

In [9]:
retrieve(
    query=(
        "What was JPMorgan Chase's Common Equity Tier 1 capital ratio "
        "at December 31, 2025?"
    ),
    ticker="JPM",
)


{'rank': 1, 'score': 0.7987838983535767, 'record_id': 'table::424e28319ce861eb0c62c04cf4a4ec6849fac523b887cc14ee9c9470cee737f2', 'target_chunk_id': '653d8ffbdc0646ee96d37ea57afb761f7d4ac1a01711e4b7bfefb0be0b9cabc3', 'record_type': 'table', 'ticker': 'JPM', 'sec_item': 'Item 15', 'section_title': 'FORWARD-LOOKING STATEMENTS'}

Embedding text:
Bank: JPM
Report: 2025 10-K
SEC item: Item 15
Section: FORWARD-LOOKING STATEMENTS
Columns: December 31, 2024 (in millions, except ratios); Standardized; Advanced; JPMorgan Chase & Co.; JPMorgan Chase Bank, N.A.; Risk-based capital metrics: (a)
Rows: CET1 capital; Tier 1 capital; Total capital; Risk-weighted assets; CET1 capital ratio; Tier 1 capital ratio; Total capital ratio
Units: in millions; percent

Original document:
December 31, 2024 (in millions, except ratios) | Standardized |  | Advanced |
JPMorgan Chase & Co. | JPMorgan Chase Bank, N.A. |  | JPMorgan Chase & Co. |  | JPMorgan Chase Bank, N.A. |
Risk-based capital metrics: (a) |  |  |  |

[{'rank': 1,
  'score': 0.7987838983535767,
  'record_id': 'table::424e28319ce861eb0c62c04cf4a4ec6849fac523b887cc14ee9c9470cee737f2',
  'target_chunk_id': '653d8ffbdc0646ee96d37ea57afb761f7d4ac1a01711e4b7bfefb0be0b9cabc3',
  'record_type': 'table',
  'ticker': 'JPM',
  'sec_item': 'Item 15',
  'section_title': 'FORWARD-LOOKING STATEMENTS'},
 {'rank': 2,
  'score': 0.7486634254455566,
  'record_id': 'table::8f4ee5cffc77f7197d3f9dc7dc187b7df3feeb63770c26a2e5b5d93cf1c18fa1',
  'target_chunk_id': '5afa4022f4b0a8902e589a8f4b095d949e8995f22454ce1f4ab374cb072e0054',
  'record_type': 'table',
  'ticker': 'JPM',
  'sec_item': 'Item 15',
  'section_title': 'Item 15. Exhibits, Financial Statement Schedules.'},
 {'rank': 3,
  'score': 0.7390989661216736,
  'record_id': 'table::4b38c88acf776302056433e9df29328081f1f6a118883156901eb830eb4e335c',
  'target_chunk_id': 'a23caa9bb077e76d30d90987c508234d9fa95a27fb8b84b856ad80c897791e4d',
  'record_type': 'table',
  'ticker': 'JPM',
  'sec_item': 'Item 15'

In [10]:
results = retrieve(
    query=(
        "What operational risks does JPMorgan face from cyberattacks "
        "and failures of technology systems?"
    ),
    ticker="JPM",
    k=3,
)

for result in results:
    record_index = record_ids.index(result["record_id"])
    print(f"\nRank {result['rank']} | Score: {result['score']:.4f}")
    print(records[record_index]["document"][:1500])


{'rank': 1, 'score': 0.8119903802871704, 'record_id': 'text::bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27', 'target_chunk_id': 'bb5849a93346079c80657d456bae84ebc362d25455705e4b8fee080db67dbe27', 'record_type': 'text', 'ticker': 'JPM', 'sec_item': 'Item 1A', 'section_title': 'Item 1A. Risk Factors.'}

Embedding text:
Bank: JPM
Report: 2025 10-K
SEC item: Item 1A
Section: Item 1A. Risk Factors.

JPMorganChase’s interconnectivity with clients, customers and other external parties continues to expand, which increases the risk of failure or cyber attacks with respect to the systems of those parties. Any systems failure, security breach, or human error or misconduct that affects clients, customers or external parties could require JPMorganChase to take steps to protect the integrity of its own operational systems or to safeguard confidential information, including restricting the access of its customers to thei

Original document:
JPMorganChase’s interconnectivity with c

In [1]:
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
import json
from pathlib import Path

import numpy as np


root = Path.cwd()

records_path = root / (
    "data/processed/embedding_records/"
    "sec_10k_embedding_records.jsonl"
)
embeddings_path = root / (
    "data/processed/embeddings/"
    "qwen3_embedding_0_6b_records.npz"
)
proxies_path = root / (
    "data/processed/table_proxies/"
    "sec_10k_table_proxies.jsonl"
)

for path in (records_path, embeddings_path, proxies_path):
    if not path.exists():
        raise FileNotFoundError(path)


def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def get_field(record: dict, field: str):
    if field in record:
        return record[field]

    metadata = record.get("metadata", {})
    if isinstance(metadata, dict):
        return metadata.get(field)

    return None


records = load_jsonl(records_path)
proxies = load_jsonl(proxies_path)

print("===== EMBEDDING RECORDS =====")
print(f"Records: {len(records)}")
print(f"First record fields: {sorted(records[0])}")

record_types = Counter(
    get_field(record, "record_type")
    for record in records
)
tickers = Counter(
    get_field(record, "ticker")
    for record in records
)
counts_by_ticker_and_type = Counter(
    (
        get_field(record, "ticker"),
        get_field(record, "record_type"),
    )
    for record in records
)

print(f"Record types: {dict(sorted(record_types.items()))}")
print(f"Tickers: {dict(sorted(tickers.items()))}")

print("\nCounts by ticker and type:")
for key, count in sorted(counts_by_ticker_and_type.items()):
    print(f"{key}: {count}")

record_ids = [
    str(get_field(record, "record_id"))
    for record in records
]
target_chunk_ids = [
    str(get_field(record, "target_chunk_id"))
    for record in records
]

print(f"\nUnique record IDs: {len(set(record_ids))}")
print(
    "Unique target chunk IDs: "
    f"{len(set(target_chunk_ids))}"
)

print("\n===== TABLE PROXIES =====")
proxy_versions = Counter(
    proxy.get("proxy_version", "<missing>")
    for proxy in proxies
)

print(f"Proxies: {len(proxies)}")
print(f"Proxy versions: {dict(proxy_versions)}")
print(f"First proxy fields: {sorted(proxies[0])}")

print("\n===== NPZ =====")
with np.load(embeddings_path, allow_pickle=False) as archive:
    print(f"NPZ keys: {archive.files}")

    for key in archive.files:
        array = archive[key]
        print(
            f"{key}: shape={array.shape}, "
            f"dtype={array.dtype}"
        )

    vector_keys = [
        key
        for key in archive.files
        if archive[key].ndim == 2
        and np.issubdtype(archive[key].dtype, np.floating)
    ]

    if len(vector_keys) != 1:
        raise ValueError(
            f"Expected one embedding matrix, found: {vector_keys}"
        )

    vector_key = vector_keys[0]
    embeddings = archive[vector_key]

    id_keys = [
        key
        for key in archive.files
        if "id" in key.lower()
        and archive[key].ndim == 1
        and len(archive[key]) == len(records)
    ]

    print(f"Embedding array key: {vector_key}")
    print(f"Possible ID keys: {id_keys}")

    norms = np.linalg.norm(embeddings, axis=1)

    print(f"Embedding count: {embeddings.shape[0]}")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    print(f"Minimum norm: {norms.min():.8f}")
    print(f"Maximum norm: {norms.max():.8f}")
    print(f"NaN values: {int(np.isnan(embeddings).sum())}")
    print(f"Inf values: {int(np.isinf(embeddings).sum())}")

    if id_keys:
        npz_ids = archive[id_keys[0]].astype(str).tolist()
        print(
            "NPZ and JSONL ID order match: "
            f"{npz_ids == record_ids}"
        )

print("\n===== VERIFICATION ENVIRONMENT =====")
packages = [
    "numpy",
    "sentence-transformers",
    "transformers",
    "torch",
    "huggingface-hub",
]

for package in packages:
    try:
        package_version = version(package)
    except PackageNotFoundError:
        package_version = "<not installed>"

    print(f"{package}: {package_version}")

===== EMBEDDING RECORDS =====
Records: 7110
First record fields: ['document', 'embedding_text', 'metadata', 'record_id', 'record_type', 'target_chunk_id']
Record types: {'table': 3220, 'text': 3890}
Tickers: {'ALLY': 786, 'BAC': 900, 'C': 1015, 'GS': 1151, 'JPM': 1423, 'LOB': 497, 'PNC': 628, 'TFC': 520, 'USB': 97, 'WFC': 93}

Counts by ticker and type:
('ALLY', 'table'): 247
('ALLY', 'text'): 539
('BAC', 'table'): 508
('BAC', 'text'): 392
('C', 'table'): 454
('C', 'text'): 561
('GS', 'table'): 531
('GS', 'text'): 620
('JPM', 'table'): 765
('JPM', 'text'): 658
('LOB', 'table'): 176
('LOB', 'text'): 321
('PNC', 'table'): 263
('PNC', 'text'): 365
('TFC', 'table'): 234
('TFC', 'text'): 286
('USB', 'table'): 19
('USB', 'text'): 78
('WFC', 'table'): 23
('WFC', 'text'): 70

Unique record IDs: 7110
Unique target chunk IDs: 7110

===== TABLE PROXIES =====
Proxies: 3220
Proxy versions: {'deterministic-v1': 3220}
First proxy fields: ['element_type', 'proxy_id', 'proxy_text', 'proxy_version', 're